# 연금복권720 당첨번호 통계 분석

- 조(1~5)별 당첨 빈도
- 자리별(1~7자리) 숫자 출현 빈도
- 자리별 숫자 분포 히트맵

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path('..') / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from crawl_pension import crawl

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA = Path('..') / 'data' / 'pension720.csv'

## 1. 데이터 수집

In [ ]:
if DATA.exists():
    df = pd.read_csv(DATA)
    print(f'기존 데이터 로드: {len(df)}회차 ({df["date"].min()} ~ {df["date"].max()})')
else:
    print('데이터 수집 중...')
    df = crawl(start=1)
    DATA.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(DATA, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {len(df)}행')

df.head()

## 2. 조(Group)별 당첨 빈도

In [ ]:
group_freq = df['group'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(group_freq.index.astype(str), group_freq.values, color='#e67e22')
ax.set_xlabel('조')
ax.set_ylabel('당첨 횟수')
ax.set_title('연금복권720 조별 당첨 빈도')
for i, v in enumerate(group_freq.values):
    ax.text(i, v + 0.3, f'{v/len(df)*100:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. 자리별 숫자 출현 빈도 히트맵

In [ ]:
num_cols = ['n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7']

# 10×7 행렬: 행=숫자(0~9), 열=자리(1~7)
heatmap = np.zeros((10, 7), dtype=int)
for col_idx, col in enumerate(num_cols):
    for digit, cnt in df[col].value_counts().items():
        heatmap[int(digit)][col_idx] = cnt

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(heatmap, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[f'{i+1}자리' for i in range(7)],
            yticklabels=list(range(10)))
ax.set_ylabel('숫자')
ax.set_title('연금복권720 자리별 숫자 출현 횟수')
plt.tight_layout()
plt.show()

## 4. 자리별 출현 빈도 막대 (각 자리 0~9 분포)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=False)
axes = axes.flatten()

for idx, col in enumerate(num_cols):
    freq = df[col].value_counts().sort_index()
    axes[idx].bar(freq.index.astype(str), freq.values, color='#3498db')
    axes[idx].set_title(f'{idx+1}자리')
    axes[idx].set_xlabel('숫자')

axes[-1].set_visible(False)  # 8번째 칸 비우기
fig.suptitle('연금복권720 자리별 숫자 출현 빈도', fontsize=13)
plt.tight_layout()
plt.show()